# Fine-tune Laya as a tier router (1 GPU: Google Colab or Kaggle)

Single-GPU adaptation of the official notebook
`NandhaKishorM/laya/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb`.
Every `laya.*` function used here appears in that notebook.

**Colab:** Runtime -> Change runtime type -> **T4 GPU**. Upload `data/labels.csv` and
`data/laya_split.csv` with the folder icon on the left.

**Kaggle:** Settings -> Accelerator -> **GPU T4 x2** (one GPU is used) and Internet **On**
(needs a phone-verified account). Add Input -> Upload -> New dataset with the same two files.

**Fairness:** trains on `laya_train` only (85% of TRAIN). Validation (15% of TRAIN) is held out
for calibration checks and the confidence threshold. TEST is never touched here.

**Output:** `laya_checkpoint.zip` -> unzip into `llm-router/routers/laya_checkpoint/`,
then run `python routers/laya_router.py` locally to produce fine-tuned predictions.

In [ ]:
# Find the uploaded files and pick an output folder (works on Colab and Kaggle)
import glob, os
ON_KAGGLE = os.path.exists("/kaggle")
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name) + glob.glob(f"/content/{name}")
    if not hits:
        raise FileNotFoundError(f"{name} not found - upload it first (see instructions above)")
    return hits[0]
LABELS_CSV, SPLIT_CSV = find("labels.csv"), find("laya_split.csv")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
OUTPUT_DIR = f"{WORK}/laya_checkpoint"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab", "|", LABELS_CSV, "|", SPLIT_CSV, "| output:", OUTPUT_DIR)

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub accelerate
import laya, torch
print("Laya", laya.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Same question definition as routers/laya_router.py with ACTIVE_TIERS=small,large (keep them identical).
# For 3 tiers, add: "medium_model": "moderate knowledge or a few reasoning steps"
QUESTIONS = {
    "tier": {
        "type": "choice",
        "instructions": "Which is the cheapest language model that will still answer this question correctly?",
        "criteria": {
            "small_model": "simple, short factual recall, an easy well-known fact, or a simple question",
            "large_model": "multi-step math, hard reasoning, or rare specialist knowledge",
        },
    }
}

## Check the gold format
`items_from_rows` expects rows of `{state, questions, gold}`. We copy the gold layout from the
official dataset, so print one real row and compare it with ours before training.

In [ ]:
import json
from datasets import load_dataset
ref = load_dataset("LocalLLaMA/typed-decisions", "all", split="train[:1]")[0]
ref_q, ref_g = json.loads(ref["questions"]), json.loads(ref["gold"])
choice_q = next(k for k, v in ref_q.items() if v["type"] == "choice")
print("official choice gold example:", ref_g[choice_q])

In [ ]:
import pandas as pd

labels = pd.read_csv(LABELS_CSV).merge(pd.read_csv(SPLIT_CSV), on="id")
train = labels[labels.laya_split == "laya_train"]
print("train label counts:", train.label.value_counts().to_dict())

# Class imbalance: oversample every tier up to the size of the largest tier
target = train.label.value_counts().max()
balanced = pd.concat([g.sample(target, replace=True, random_state=42) for _, g in train.groupby("label")])
balanced = balanced.sample(frac=1, random_state=42)
print("after oversampling:", balanced.label.value_counts().to_dict())

def gold_for(label):
    keys = list(QUESTIONS["tier"]["criteria"])
    k = f"{label}_model"
    return {"tier": {"label": k, "probabilities": {x: (1.0 if x == k else 0.0) for x in keys}}}

rows = [{"state": {"question": q}, "questions": QUESTIONS, "gold": gold_for(l)}
        for q, l in zip(balanced.question, balanced.label)]
print("our gold example:", rows[0]["gold"]["tier"])

In [ ]:
import os, json
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

model_dir = snapshot_download("convaiinnovations/laya")
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))

items, skipped = items_from_rows(tok, rows, cfg["max_len"], cfg["head_max_len"])
print(f"{len(items)} training items, {skipped} skipped")

## Train (single GPU version of the official DDP loop)

In [ ]:
import math, random, time
from safetensors.torch import load_file
from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import calibration_records, encode_item, rlcd_loss, save_checkpoint, sigma_at, split_calibration

device = torch.device("cuda")
cfg["gradient_checkpointing"] = True
max_len, head_max_len = cfg["max_len"], cfg["head_max_len"]

model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.head_checkpointing = True
model.to(device).train()

# Same helper as the official notebook: holds out a slice for temperature calibration
train_items, calib_items = split_calibration(items, calib_max=400, calib_frac=0.1, seed=20260922)

EPOCHS, MICRO_BATCH, GRAD_ACCUM, GROUP_SIZE = 4, 8, 4, 4
LR_ENCODER, LR_HEAD, SIGMA_START, SIGMA_END = 2.5e-5, 1.0e-4, 0.4, 0.1

enc = [p for n, p in model.named_parameters() if "encoder." in n]
head = [p for n, p in model.named_parameters() if "encoder." not in n]
opt = torch.optim.AdamW([{"params": enc, "lr": LR_ENCODER}, {"params": head, "lr": LR_HEAD}], weight_decay=0.01)
total_updates = max(1, (len(train_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_updates, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda")

t0 = time.time()
for epoch in range(EPOCHS):
    random.seed(42 + epoch)
    random.shuffle(train_items)
    sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
    opt.zero_grad(set_to_none=True)
    total, n = 0.0, 0
    n_steps = math.ceil(len(train_items) / MICRO_BATCH)
    for step, b in enumerate(range(0, len(train_items), MICRO_BATCH)):
        enc_items = [encode_item(tok, it, max_len, head_max_len) for it in train_items[b:b + MICRO_BATCH]]
        batch = collate_items([enc_items], tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                              batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                              batch["qtype"].to(device))
        loss = rlcd_loss(logits.float(), batch["target"].to(device), batch["marker_mask"].to(device),
                         batch["qtype"].to(device), sigma, samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)
        scaler.scale(loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == n_steps:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
        total, n = total + loss.item(), n + 1
    print(f"epoch {epoch + 1}/{EPOCHS}  avg loss {total / n:.4f}  ({time.time() - t0:.0f}s)")

In [ ]:
# Temperature calibration + save (as in the official notebook)
model.eval()
out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len, fine_tuned=True, model_name="laya-tier-router")
try:
    fitted = fit_temperature_map(calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16))
    out_cfg["temperature"] = fitted["temperature"]
    out_cfg.pop("temperature_by_options", None)
    if fitted["temperature_by_options"]:
        out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
except Exception as e:
    print("Temperature fitting skipped:", e)
save_checkpoint(model, tok, out_cfg, OUTPUT_DIR)
print("saved to", OUTPUT_DIR)

## Quick check on VALIDATION (not test)

In [ ]:
agent = laya.Agent(OUTPUT_DIR, device="cuda")
val = labels[labels.laya_split == "validation"]
pred = [agent.predict({"question": q}, QUESTIONS)["answers"]["tier"]["choice"].replace("_model", "") for q in val.question]
print("validation routing accuracy:", (pd.Series(pred, index=val.index) == val.label).mean())
print(pd.crosstab(val.label, pd.Series(pred, index=val.index, name="pred")))

In [ ]:
# Zip the checkpoint for download
import shutil
zip_path = shutil.make_archive(f"{WORK}/laya_checkpoint", "zip", WORK, "laya_checkpoint")
print("created", zip_path)
if ON_KAGGLE:
    print("Kaggle: download laya_checkpoint.zip from the Output panel on the right (/kaggle/working)")
else:
    from google.colab import files
    files.download(zip_path)